In [1]:
!pip install transformers datasets torch scikit-learn


In [2]:
pip install --upgrade transformers datasets accelerate

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import re
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset, ClassLabel

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from torch.nn import CrossEntropyLoss

In [4]:
df = pd.read_csv("data/Resume.csv")
df = df[["Resume_str", "Category"]]

In [5]:
def map_domain(category):
    category = category.lower()

    if category in ["accountant", "finance", "banking", "bpo"]:
        return "BUSINESS"
    elif category in ["engineering", "information-technology", "computer-science"]:
        return "TECH"
    elif category in ["arts", "design", "fashion", "media"]:
        return "CREATIVE"
    elif category in ["healthcare", "medical", "pharmacy"]:
        return "HEALTH"
    elif category in ["advocate", "law"]:
        return "LEGAL_ADMIN"
    elif category in ["teacher", "education"]:
        return "EDUCATION"
    else:
        return "OTHER"

df["label"] = df["Category"].apply(map_domain)

In [6]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[^a-zA-Z ]", " ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["Resume_str"].apply(clean_text)
df = df[["text", "label"]]

In [7]:
# Reduce OTHER (very important)
other_df = df[df["label"] == "OTHER"].sample(n=250, random_state=42)

rest_df = df[df["label"] != "OTHER"]

df = pd.concat([other_df, rest_df])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df["label"].value_counts())

label
BUSINESS       373
OTHER          250
TECH           238
LEGAL_ADMIN    118
HEALTH         115
CREATIVE       103
EDUCATION      102
Name: count, dtype: int64


In [8]:
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])

num_labels = len(le.classes_)
print(le.classes_)

['BUSINESS' 'CREATIVE' 'EDUCATION' 'HEALTH' 'LEGAL_ADMIN' 'OTHER' 'TECH']


In [9]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(df["label_encoded"]),
    y=df["label_encoded"]
)

class_weights = torch.tensor(class_weights, dtype=torch.float)

In [10]:
dataset = Dataset.from_pandas(df[["text", "label_encoded"]])
dataset = dataset.rename_column("label_encoded", "labels")

dataset = dataset.cast_column(
    "labels",
    ClassLabel(num_classes=num_labels, names=list(le.classes_))
)

Casting the dataset:   0%|          | 0/1299 [00:00<?, ? examples/s]

In [11]:
dataset = dataset.train_test_split(test_size=0.2, stratify_by_column="labels")

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [12]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256   # instead of 512   # 🔥 CRITICAL
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")

Map:   0%|          | 0/1039 [00:00<?, ? examples/s]

Map:   0%|          | 0/260 [00:00<?, ? examples/s]

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=10,
    save_total_limit=2,
    fp16=False,
    report_to="none" 
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [16]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [17]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = CrossEntropyLoss(weight=class_weights.to(model.device))
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [18]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)
import torch
torch.cuda.empty_cache()

trainer.train()

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.871102,0.844757,0.784615,0.765616
2,0.542223,0.576577,0.850000,0.853728
3,0.211756,0.579272,0.850000,0.852541
4,0.313892,0.661874,0.861538,0.863707
5,0.115922,0.638191,0.857692,0.859795


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1300, training_loss=0.6557223126750726, metrics={'train_runtime': 380.1597, 'train_samples_per_second': 13.665, 'train_steps_per_second': 3.42, 'total_flos': 344114749263360.0, 'train_loss': 0.6557223126750726, 'epoch': 5.0})

In [19]:
model.save_pretrained("models/resume_classifier_model")
tokenizer.save_pretrained("models/resume_classifier_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('models/resume_classifier_model/tokenizer_config.json',
 'models/resume_classifier_model/tokenizer.json')

In [20]:
import joblib
joblib.dump(le, "models/label_encoder.pkl")

['models/label_encoder.pkl']

In [1]:
import json

with open("models/label_mapping.json", "w") as f:
    json.dump({i: label for i, label in enumerate(le.classes_)}, f)

NameError: name 'le' is not defined